In [44]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix, save_npz, load_npz
from sklearn.metrics.pairwise import cosine_similarity
import os
import kagglehub

In [45]:
DATA_DIR = os.path.join("..", "data")  # same relative-path pattern as before

train = pd.read_parquet(os.path.join(DATA_DIR, "train_ratings.parquet"))
test = pd.read_parquet(os.path.join(DATA_DIR, "test_ratings.parquet"))

train_sample = train.sample(1_000_000, random_state=42)
test_sample = test.sample(200_000, random_state=42)

print(train_sample.shape, test_sample.shape)

(1000000, 4) (200000, 4)


## Baseline 1 (K most popular shows)

In [46]:
# Count positive interactions per anime, using train only
popularity = train_sample[train_sample['is_positive'] == 1].groupby('anime_id').size().sort_values(ascending=False)

#K most popular shows
K = 10
top_k_popular = popularity.head(K).index.tolist()

print(top_k_popular)

#Find the precision and recall rate
def precision_recall_at_k(test_df, recommended_items, k):
    total_relevant = 0
    total_recommended_relevant = 0
    
    for user_id, group in test_df[test_df['is_positive'] == 1].groupby('user_id'):
        actual_positive = set(group['anime_id'])
        recommended = set(recommended_items[:k])
        
        hits_this_user = len(actual_positive & recommended)
        total_recommended_relevant += hits_this_user
        total_relevant += len(actual_positive)
    
    # Recall: of everything the user actually liked, what fraction did we recommend?
    recall = total_recommended_relevant / total_relevant if total_relevant > 0 else 0
    
    # Precision: of everything we recommended, what fraction did users actually like?
    num_users = test_df['user_id'].nunique()
    precision = total_recommended_relevant / (num_users * k)
    
    return precision, recall

precision, recall = precision_recall_at_k(test_sample, top_k_popular, K)
print(f"Recall@{K}: {recall:.4f}")
print(f"Precision@{K}: {precision:.4f}")


[20, 2, 92, 2376, 99, 1147, 100, 726, 1154, 1167]
Recall@10: 0.0657
Precision@10: 0.0045


## Baseline 2

In [53]:
anime_ids = train_sample['anime_id'].astype('category')
anime_id_map = dict(enumerate(anime_ids.cat.categories))      
anime_id_map_reverse = {v: k for k, v in anime_id_map.items()} 

user_ids = train_sample['user_id'].astype('category')
user_id_map = dict(enumerate(user_ids.cat.categories))
user_id_map_reverse = {v: k for k, v in user_id_map.items()}

train_sample['anime_idx'] = anime_ids.cat.codes
train_sample['user_idx'] = user_ids.cat.codes

test_sample['anime_idx'] = test_sample['anime_id'].map(anime_id_map_reverse)
test_sample['user_idx'] = test_sample['user_id'].map(user_id_map_reverse)


p_train = train_sample[train_sample['is_positive']==1]
test_sample_clean = test_sample.dropna(subset=['anime_idx', 'user_idx'])

In [54]:
display(test_sample_clean)

,user_id,anime_id,rating,is_positive,anime_idx,user_idx
27589591,614249,2897,1,0,2891.0,121081.0
89514452,1169341,3361,8,1,3353.0,318949.0
72320556,1023503,1812,8,1,1808.0,263922.0
111001780,1386170,5300,10,1,5285.0,393378.0
74834209,1042731,3059,9,1,3053.0,271689.0
...,...,...,...,...,...,...
47053046,820767,1270,10,1,1269.0,184945.0
64074669,957680,7,9,1,6.0,238188.0
130368837,1575190,225,10,1,224.0,460140.0
107848125,1355436,11286,10,1,10562.0,382680.0


In [55]:
item_user_matrix = csr_matrix(
    (
    [1] * len(p_train), (p_train['anime_idx'], p_train['user_idx'])
    ),
    shape = (len(anime_id_map), len(user_id_map))
)

item_similarity = cosine_similarity(item_user_matrix, dense_output=False)
save_npz(os.path.join(DATA_DIR, "item_user_matrix.npz"), item_user_matrix)
save_npz(os.path.join(DATA_DIR, "item_similarity.npz"), item_similarity)

test_sample['anime_idx'] = anime_ids.cat.codes
test_sample['user_idx'] = user_ids.cat.codes

In [56]:
def recommend_for_user(user_idx, k=10):
    # Get this user's positively-rated anime (as matrix indices)
    user_rated = p_train[p_train['user_idx'] == user_idx]['anime_idx'].tolist()
    
    if not user_rated:
        return []  
    

    scores = np.asarray(item_similarity[user_rated].sum(axis=0)).flatten()
    
    scores[user_rated] = -1
    
    top_idx = scores.argsort()[::-1][:k]
    
    return [anime_id_map[i] for i in top_idx]

sample_user_idx = 5
recs = recommend_for_user(sample_user_idx, k=10)

path = kagglehub.dataset_download("ramazanturann/user-animelist-dataset")
animes = pd.read_csv(os.path.join(path, "animes.csv"))

print(f'10 Recommend animes: \n\n {animes[animes['animeID'].isin(recs)][['animeID', 'title']]}')

10 Recommend animes: 

       animeID                                  title
340       341                              Geneshaft
433       434                             Moyashimon
758       759                             Ani*Kuri15
995       996                Momo, Girl God of Death
2000     2001             Bakuretsu Tenshi: Infinity
3597     3598        Miss Monochrome The Animation 3
4662     4663  The Super Dimension Century Orguss 02
4918     4919               Star☆Twinkle Pretty Cure
5807     5808                                 Inu-Oh
6893     6894                              Mini Yuri


In [ ]:
print(test_sample_clean.shape)
print(test_sample_clean['is_positive'].value_counts())
print(test_sample_clean['user_idx'].isna().sum())

In [58]:
def precision_recall_itemcf(test_df, k=10, sample_users=None):
    total_relevant = 0
    total_recommended_relevant = 0
    
    test_positive = test_df[test_df['is_positive'] == 1]
    grouped = test_positive.groupby('user_idx')
    
    users_to_eval = list(grouped.groups.keys())
    if sample_users:
        users_to_eval = np.random.choice(users_to_eval, size=sample_users, replace=False)
    
    for user_idx in users_to_eval:
        group = grouped.get_group(user_idx)
        actual_positive = set(group['anime_id'])
        
        recommended = set(recommend_for_user(user_idx, k=k))  # ← computed fresh, per user
        
        hits_this_user = len(actual_positive & recommended)
        total_recommended_relevant += hits_this_user
        total_relevant += len(actual_positive)
    
    recall = total_recommended_relevant / total_relevant if total_relevant > 0 else 0
    precision = total_recommended_relevant / (len(users_to_eval) * k)
    
    return precision, recall

precision_recall_itemcf(test_sample_clean, k=10, sample_users=10)

ValueError: 'a' cannot be empty unless no samples are taken